In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import seaborn as sns
import pandas as pd
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))
from regression_modelling.models.cv import (
    full_coverage_cities, load_pooled_table, loco_folds,
)

In [2]:
# Load 5 cities with comprehensive crime incident coverage
cities = full_coverage_cities()            # ['houston','chicago','atlanta','kansas_city','detroit']
# daytime_pop_floor=100 (default) drops small-denominator rate ARTIFACTS — a few crimes
# over ~no daytime people — the diagnosed target-stabilization treatment. Genuine hotspots
# (hundreds of crimes over a solid daytime population) sit far above the floor and are kept.
pooled = load_pooled_table(daytime_pop_floor=100)

# inspect the LOCO split
for city, train, holdout in loco_folds(pooled):
    print(city, "→ train", train.shape, "holdout", holdout.shape)


  houston            1629 BGs
  chicago            2164 BGs
  atlanta             426 BGs
  kansas_city         473 BGs
  detroit             622 BGs
  pooled             5314 BGs across 5 cities
  dropped 22 zero/NaN-pop BGs -> 5292 remain (filtered geoid set for fit + Moran's I + bias join)
  dropped 8 BGs below daytime_pop 100 (small-denominator rate artifacts) -> 5284 remain
atlanta → train (4860, 82) holdout (424, 82)
chicago → train (3125, 82) holdout (2159, 82)
detroit → train (4679, 82) holdout (605, 82)
houston → train (3657, 82) holdout (1627, 82)
kansas_city → train (4815, 82) holdout (469, 82)


### Component 2 — leakage-safe fit / predict core

One design matrix, three target forms. The predictor scaler is fit on the **train cities
only** and applied to the holdout, so no holdout information touches the fit.

- `rate_within_city` — **headline (c)**: per-city z-scored rate (relative within-city risk)
- `rate` — **reported (a)**: raw rate (absolute level)
- `logcount` — comparator: log(count+1)

Demonstrated below on a **single fold** (hold out one city). The full LOCO loop is the next
component.

In [3]:
from scipy.stats import spearmanr
from regression_modelling.models.cv import (
    fit_fold, predict_fold, make_target, TARGET_MODES,
)

# one fold: hold out a chosen city, train on the other four
HOLDOUT = "chicago"
folds = {c: (tr, ho) for c, tr, ho in loco_folds(pooled)}
train, holdout = folds[HOLDOUT]
print(f"holdout={HOLDOUT}  train={train.shape}  holdout={holdout.shape}")

holdout=chicago  train=(3125, 82)  holdout=(2159, 82)


In [4]:
# fit each target mode on the train fold, score the held-out city
rows = []
scored_by_mode = {}
for mode in TARGET_MODES:
    fit = fit_fold(train, mode=mode)
    scored = predict_fold(fit, holdout)
    scored_by_mode[mode] = scored
    rows.append({
        "mode": mode,
        "n_train": fit["n_train"],
        "adj_r2_in_sample": round(fit["result"].rsquared_adj, 3),
        "holdout_n": len(scored),
        # ranking sanity: predicted risk vs ACTUAL holdout rate (headline metric is rank-based)
        "spearman_pred_vs_rate": round(spearmanr(scored["y_pred"], scored["cl_total_rate"]).statistic, 3),
    })
pd.DataFrame(rows).set_index("mode")

,n_train,adj_r2_in_sample,holdout_n,spearman_pred_vs_rate
mode,,,,
rate_within_city,2950,0.053,2064,0.323
rate,2950,0.022,2064,0.257
rate_daytime_within_city,2950,0.172,2064,0.576
rate_daytime,2950,0.224,2064,0.578
logcount,2950,0.375,2064,0.532


In [5]:
# peek at the headline (within-city) fold's top predicted-risk BGs in the held-out city
scored_by_mode["rate_within_city"][["geoid", "city", "population", "cl_total_count",
                                      "cl_total_rate", "y_pred"]] \
    .sort_values("y_pred", ascending=False).head(10)

,geoid,city,population,cl_total_count,cl_total_rate,y_pred
3689,170318391001,chicago,4975.0,1581.0,317.788945,1.498123
2972,170315001004,chicago,1458.0,65.0,44.581619,1.083718
3791,170319801001,chicago,18.0,130.0,7222.222222,1.071960
3027,170315401012,chicago,2464.0,119.0,48.295455,0.931048
3762,170318428004,chicago,1106.0,63.0,56.962025,0.903123
2983,170315103002,chicago,1050.0,150.0,142.857143,0.864228
3129,170316113002,chicago,1064.0,15.0,14.097744,0.860803
2887,170314602001,chicago,779.0,39.0,50.064185,0.834926
3597,170318340001,chicago,2777.0,73.0,26.287360,0.817210
3614,170318346002,chicago,1151.0,124.0,107.732407,0.805929


### Component 3 — LOCO driver (all 5 folds)

Rotating leave-one-city-out for each target mode: fit on the four train cities, score the
held-out city, repeat. `run_all_modes` returns `{mode: result}` where each result carries
`scored` (every BG's **out-of-sample** prediction) and per-fold `fits`.

In [6]:
from regression_modelling.models.cv import run_loco, run_all_modes, loco_metrics, plot_lorenz

# winsor_upper=250 gently caps whatever the daytime floor leaves in the top tail (a fixed
# constant, so it is leakage-free). run_all_modes now spans BOTH denominators:
#   rate_within_city, rate                 (plain population rate)
#   rate_daytime_within_city, rate_daytime (population + LODES jobs)  + logcount comparator
runs = run_all_modes(pooled, winsor_upper=250)


LOCO — target mode = 'rate_within_city' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.320
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.302


  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.342
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.325


  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.302
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate' (cl_total), winsor@250
  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.342
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.313


  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.345
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.356
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.321
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate_daytime_within_city' (cl_total), winsor@250


  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.250
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.177


  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.281
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.301
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.243
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'rate_daytime' (cl_total), winsor@250


  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.287
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.233
  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.268


  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.364
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.276
  -> 5014 BGs scored out-of-sample across 5 folds
LOCO — target mode = 'logcount' (cl_total), winsor@250


  holdout=atlanta        n_train= 4645 scored=  369 adjR2_in= 0.401
  holdout=chicago        n_train= 2950 scored= 2064 adjR2_in= 0.375


  holdout=detroit        n_train= 4414 scored=  600 adjR2_in= 0.412
  holdout=houston        n_train= 3478 scored= 1536 adjR2_in= 0.428
  holdout=kansas_city    n_train= 4569 scored=  445 adjR2_in= 0.390
  -> 5014 BGs scored out-of-sample across 5 folds


### Component 4 — held-out metrics

**Headline = concentration/Lorenz.** `gini` = model concentration (0 = no skill), `gini_oracle`
= best achievable, `skill` = gini/gini_oracle, `capture@20` = share of crime in the highest-risk
BGs covering 20% of population. `spearman` = rank corr of predicted score vs actual rate.
R²/RMSE/MAE appear only for the absolute `rate` mode.

In [7]:
# headline target: within-city standardized rate, population-weighted x-axis
loco_metrics(runs["rate_within_city"], x_unit="population")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.293,0.463,0.633,0.363,0.672
chicago,2064,0.289,0.445,0.649,0.381,0.582
detroit,600,0.160,0.292,0.549,0.274,0.475
houston,1536,0.310,0.520,0.596,0.386,0.557
kansas_city,445,0.375,0.517,0.726,0.454,0.685
POOLED,5014,0.279,0.470,0.593,0.360,0.560


In [8]:
# reported target (a): plain pooled raw rate — adds out-of-sample R2/RMSE/MAE
loco_metrics(runs["rate"], x_unit="population")

,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.298,0.463,0.643,0.364,0.683,-0.436,56.88,46.14
chicago,2064,0.295,0.445,0.663,0.386,0.602,0.024,174.95,24.80
detroit,600,0.159,0.292,0.546,0.274,0.471,0.005,367.77,50.59
houston,1536,0.318,0.520,0.613,0.386,0.578,0.000,3246.87,117.53
kansas_city,445,0.385,0.517,0.746,0.454,0.711,0.006,418.14,50.94
POOLED,5014,0.289,0.470,0.614,0.368,0.586,0.001,1809.44,60.18


In [9]:
# comparator: log(count+1)
loco_metrics(runs["logcount"], x_unit="population")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.283,0.463,0.611,0.357,0.652
chicago,2064,0.276,0.445,0.620,0.374,0.532
detroit,600,0.161,0.292,0.550,0.291,0.445
houston,1536,0.329,0.520,0.633,0.388,0.600
kansas_city,445,0.387,0.517,0.748,0.447,0.708
POOLED,5014,0.284,0.470,0.605,0.367,0.561


In [10]:
# concentration curves per held-out city (headline target)
plot_lorenz(runs["rate_within_city"], x_unit="population")

<Axes: title={'center': 'LOCO concentration — rate_within_city (x=population)'}, xlabel='cumulative share of population', ylabel='cumulative share of cl_total crime captured'>

#### x-axis sensitivity — population vs block-group

Population-weighting is the default (per-capita rate index). The BG-axis is the per-place
framing; compare to see how much the ranking verdict depends on the weighting choice.

In [11]:
loco_metrics(runs["rate_within_city"], x_unit="bg")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.272,0.458,0.594,0.331,0.672
chicago,2064,0.340,0.464,0.732,0.417,0.582
detroit,600,0.070,0.321,0.219,0.198,0.475
houston,1536,0.321,0.521,0.617,0.407,0.557
kansas_city,445,0.318,0.461,0.690,0.408,0.685
POOLED,5014,0.284,0.477,0.597,0.364,0.560


### Target stabilization — daytime denominator + floor + winsorize

The plain population rate is dominated by a few small-denominator artifacts (skew ≈ 32; see
`02_regression_inference`). The **daytime** denominator (population + LODES jobs) plus the
`daytime_pop ≥ 100` floor and a gentle winsorize give an interpretable rate target with a
sane scale. Score the daytime modes with the coherent `x_unit="daytime_pop"` weighting.

In [12]:
# absolute daytime rate (interpretable target) — adds out-of-sample R2/RMSE/MAE
loco_metrics(runs["rate_daytime"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.211,0.459,0.460,0.307,0.609,-3.486,48.63,41.93
chicago,2064,0.089,0.436,0.203,0.186,0.547,0.207,21.82,17.12
detroit,600,0.074,0.324,0.230,0.226,0.402,-0.039,23.14,17.89
houston,1536,0.128,0.484,0.264,0.228,0.464,0.099,29.44,18.56
kansas_city,445,0.211,0.445,0.474,0.285,0.639,0.287,22.35,15.20
POOLED,5014,0.099,0.454,0.218,0.205,0.497,-0.019,27.27,19.31


In [13]:
# within-city standardized daytime rate (relative BG risk, city level removed)
loco_metrics(runs["rate_daytime_within_city"], x_unit="daytime_pop")

,n,gini,gini_oracle,skill,capture@20,spearman
holdout,,,,,,
atlanta,369,0.210,0.459,0.458,0.307,0.601
chicago,2064,0.076,0.436,0.175,0.176,0.527
detroit,600,0.079,0.324,0.243,0.226,0.407
houston,1536,0.123,0.484,0.253,0.220,0.443
kansas_city,445,0.191,0.445,0.429,0.255,0.592
POOLED,5014,0.084,0.454,0.186,0.182,0.464


In [14]:
# concentration curves per held-out city (daytime rate)
plot_lorenz(runs["rate_daytime"], x_unit="daytime_pop")

<Axes: title={'center': 'LOCO concentration — rate_daytime (x=daytime_pop)'}, xlabel='cumulative share of block groups', ylabel='cumulative share of cl_total crime captured'>

### Model-form probe — LightGBM (linearity vs feature ceiling)

Fit a gradient-boosted tree on the **same features, same LOCO folds, same stabilized target**
as the OLS harness — only the estimator changes. Reading:

- **GBM skill ≫ OLS skill** → *linearity is the bottleneck*: the features carry signal OLS
  can't reach (nonlinearity / interactions). Add splines/interactions or go nonlinear.
- **GBM skill ≈ OLS skill** → *features are the ceiling*: the linear form is fine; get better
  features.

Rows are matched to OLS (`dropna` on predictors) so the comparison is controlled. LightGBM
also gives gain-importance now and SHAP later. NB: `n_jobs=1` — `-1` oversubscribes threads
and hangs on this VM.

In [15]:
import numpy as np
import lightgbm as lgb
from regression_modelling.models.cv import make_target
from regression_modelling.constants import PREDICTOR_COLS

LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.03, num_leaves=31,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    min_child_samples=30, reg_lambda=1.0, random_state=0,
    n_jobs=1, verbosity=-1,   # n_jobs=1: -1 hangs on this VM (thread oversubscription)
)

def run_loco_gbm(pooled, mode="rate_daytime", winsor_upper=250, params=None,
                 predictors=PREDICTOR_COLS, category="cl_total"):
    """LOCO with LightGBM — same folds/features/target as OLS, so any skill gap is model
    FORM. Returns a run dict consumable by loco_metrics / plot_lorenz."""
    params = params or LGB_PARAMS
    parts, fits = {}, {}
    scored_parts = []
    print(f"LOCO-GBM — mode={mode!r} winsor@{winsor_upper}")
    for city, train, holdout in loco_folds(pooled):
        tr = train.copy()
        tr["_y"] = make_target(tr, mode, category, winsor_upper=winsor_upper)
        tr = tr.dropna(subset=list(predictors) + ["_y"])
        ho = holdout.dropna(subset=list(predictors)).copy()
        model = lgb.LGBMRegressor(**params).fit(tr[predictors], tr["_y"])
        ho["y_pred"] = model.predict(ho[predictors])
        fits[city] = model
        scored_parts.append(ho.assign(holdout_city=city))
        print(f"  holdout={city:14} n_train={len(tr):>5} scored={len(ho):>5}")
    scored = pd.concat(scored_parts, ignore_index=True)
    return {"scored": scored, "fits": fits, "mode": mode,
            "category": category, "predictors": list(predictors)}

gbm_run = run_loco_gbm(pooled, mode="rate_daytime", winsor_upper=250)
loco_metrics(gbm_run, x_unit="daytime_pop")

LOCO-GBM — mode='rate_daytime' winsor@250


  holdout=atlanta        n_train= 4645 scored=  369


  holdout=chicago        n_train= 2950 scored= 2064


  holdout=detroit        n_train= 4414 scored=  600


  holdout=houston        n_train= 3478 scored= 1536


  holdout=kansas_city    n_train= 4569 scored=  445


,n,gini,gini_oracle,skill,capture@20,spearman,r2_oos,rmse,mae
holdout,,,,,,,,,
atlanta,369,0.143,0.459,0.312,0.240,0.511,-0.034,23.34,18.43
chicago,2064,0.188,0.436,0.431,0.317,0.601,0.284,20.73,15.05
detroit,600,0.142,0.324,0.439,0.270,0.414,0.135,21.12,16.26
houston,1536,0.163,0.484,0.337,0.265,0.396,0.082,29.73,18.58
kansas_city,445,0.228,0.445,0.513,0.283,0.618,0.218,23.40,15.85
POOLED,5014,0.171,0.454,0.376,0.293,0.539,0.192,24.28,16.60


In [16]:
# OLS vs LightGBM on the SAME daytime target/folds — pooled headline comparison
def _pooled(run, label):
    row = loco_metrics(run, x_unit="daytime_pop").loc["POOLED"]
    return {"model": label, **row[["skill", "gini", "capture@20", "spearman", "r2_oos"]].to_dict()}

pd.DataFrame([
    _pooled(runs["rate_daytime"], "OLS (linear)"),
    _pooled(gbm_run,              "LightGBM"),
]).set_index("model")

,skill,gini,capture@20,spearman,r2_oos
model,,,,,
OLS (linear),0.218,0.099,0.205,0.497,-0.019
LightGBM,0.376,0.171,0.293,0.539,0.192


In [17]:
# LightGBM gain importance, averaged across LOCO folds (precursor to SHAP)
import matplotlib.pyplot as plt
imp = (pd.DataFrame({c: m.booster_.feature_importance("gain") for c, m in gbm_run["fits"].items()},
                    index=PREDICTOR_COLS)
       .mean(axis=1).sort_values())
ax = imp.plot.barh(figsize=(8, 8), color="#2c7fb8")
ax.set_title("LightGBM mean gain importance (avg over LOCO folds)")
ax.set_xlabel("gain"); plt.tight_layout(); plt.show()